# Lab 15 — Decision Trees and Random Forests

In this lab you will fit a single decision tree and watch it overfit as you remove its depth limit, then build a random forest and see how bagging many trees together produces a more stable, more accurate model — including a hand-built mini version of the bagging mechanism itself.

**Concepts covered:** Gini impurity and greedy splitting, how tree depth controls overfitting, bagging (bootstrap resampling), out-of-bag (OOB) scoring, and random forest feature importance.

**Reference working sessions:**
- `working-sessions/supervised/07_decision_trees.ipynb`
- `working-sessions/supervised/08_random_forests.ipynb`
- `working-sessions/ml_concepts/13_interpretability_vs_complexity.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_heart_disease,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'A',
    'q3': 'B',
    'q4': 'B',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — At each node, a decision tree considers many candidate splits (a
# feature and a threshold) before picking one. What best describes how it
# chooses which split to actually use?
#
#   A) It always splits on whichever feature appears first in the dataset's
#      column order
#   B) It tries many candidate splits and keeps the one that leaves the two
#      resulting groups as pure (closest to single-class) as possible
#   C) It picks a random feature and threshold at every node to keep the
#      tree diverse
#   D) It always splits on the feature with the largest raw numeric values

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/supervised/07_decision_trees.ipynb " \
    "and the Gini/threshold-search widget in the first half of the notebook."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — An unrestricted-depth decision tree (max_depth=None) tends to have a
# much bigger gap between training accuracy and test accuracy than a shallow
# tree does. What's the best explanation for this?
#
#   A) The unrestricted tree keeps splitting until it isolates very small
#      groups of training points (sometimes single points), so it
#      memorizes noise in the training data that doesn't generalize to new
#      patients
#   B) Tree depth has no real effect on the train/test gap, only on runtime
#   C) A shallow tree always has a bigger gap because it hasn't finished
#      splitting yet
#   D) The unrestricted tree underfits because it can only make horizontal
#      splits

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/supervised/07_decision_trees.ipynb " \
    "and the max_depth widget that compares training accuracy to cross-validated accuracy."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — A random forest fits many trees, each on a bootstrap-resampled
# subset of rows, and each considering only a random subset of features at
# every split. What does this combination actually buy the forest over a
# single decision tree?
#
#   A) It guarantees every tree in the forest reaches the exact same
#      accuracy as every other tree
#   B) It decorrelates the individual trees, so their mistakes don't all
#      point the same direction — averaging their predictions then reduces
#      the variance of the final prediction compared to any single tree
#   C) It removes the need for a test set entirely
#   D) It makes each individual tree deeper and more likely to overfit

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/supervised/08_random_forests.ipynb " \
    "and the variance-reduction math in \"The math behind it\" section."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — Each tree in a random forest is trained on a bootstrap resample, so
# roughly a third of the training rows are left out of any given tree's
# resample. What is the out-of-bag (OOB) score, and why is it called a
# "free" validation estimate?
#
#   A) It's the accuracy computed on the training data itself, since every
#      row technically belongs to some tree's resample
#   B) It's each tree's accuracy evaluated only on the rows its own
#      resample happened to leave out, averaged across all trees — giving a
#      validation-like estimate without setting aside a separate validation
#      set
#   C) It's a random baseline number used only to compare against a coin flip
#   D) It's the accuracy computed after removing the least important
#      features from the model

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/supervised/08_random_forests.ipynb " \
    "and the out-of-bag scoring widget."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: What a decision tree split actually optimizes"),
    (q2_answer, _ak['q2'], "Q2: Why unrestricted depth widens the train/test gap"),
    (q3_answer, _ak['q3'], "Q3: What bagging and feature subsampling buy a random forest"),
    (q4_answer, _ak['q4'], "Q4: What the out-of-bag (OOB) score represents"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below fit a shallow tree and a deep tree to see overfitting directly, fit a random forest and inspect its OOB score and feature importances, then hand-build a tiny bagging ensemble to see the mechanism behind a random forest for yourself. Run them in order, each step builds on the previous one.

Run the setup cell first.

In [ ]:
# Shared setup — run this before the exercises
X, y = load_heart_disease()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dataset shape:", X.shape)
print("Training samples:", X_train.shape[0], " Test samples:", X_test.shape[0])

### B1 — Depth and overfitting

Fit a shallow tree (`max_depth=2`) and an unrestricted tree (`max_depth=None`) on the same data, and compare each one's training accuracy to its test accuracy.

In [ ]:
# B1 — Fit a shallow tree and a deep tree, and compare their train/test accuracy gap
shallow_tree = ___(max_depth=2, random_state=42)   # YOUR CODE — the shallow decision tree classifier for this exercise
shallow_tree.fit(___, ___)                          # YOUR CODE — the training features and training target

deep_tree = ___(max_depth=None, random_state=42)    # YOUR CODE — the unrestricted-depth decision tree classifier for this exercise
deep_tree.fit(___, ___)                             # YOUR CODE — the training features and training target

shallow_train_acc = accuracy_score(y_train, shallow_tree.predict(X_train))
shallow_test_acc = accuracy_score(___, ___)   # YOUR CODE — the true test labels and the shallow tree's predictions on the test features

deep_train_acc = accuracy_score(y_train, deep_tree.predict(X_train))
deep_test_acc = accuracy_score(___, ___)   # YOUR CODE — the true test labels and the deep tree's predictions on the test features

shallow_gap = shallow_train_acc - shallow_test_acc
deep_gap = deep_train_acc - deep_test_acc

print(f"Shallow tree (max_depth=2): train={shallow_train_acc:.3f}  test={shallow_test_acc:.3f}  gap={shallow_gap:.3f}")
print(f"Deep tree (max_depth=None): train={deep_train_acc:.3f}  test={deep_test_acc:.3f}  gap={deep_gap:.3f}")

# --- checks ---
assert deep_train_acc > 0.95, \
    "An unrestricted-depth tree should fit the training data almost perfectly"
assert deep_gap > shallow_gap, \
    "The unrestricted-depth tree should show a bigger train/test gap than the shallow tree"
print("✓ B1 complete!")

### B2 — Random forest, OOB score, and feature importance

Fit a random forest with out-of-bag scoring enabled, compare its OOB score to its actual test accuracy, then plot which features it relied on most.

In [ ]:
# B2 — Fit a random forest with OOB scoring, and inspect feature importance
forest = ___(n_estimators=200, oob_score=True, random_state=42)   # YOUR CODE — the random forest classifier for this exercise
forest.fit(___, ___)   # YOUR CODE — the training features and training target

forest_test_preds = forest.predict(X_test)
forest_test_acc = accuracy_score(___, ___)   # YOUR CODE — the true test labels and the forest's predictions on the test features

print(f"OOB score:          {forest.oob_score_:.3f}")
print(f"Test set accuracy:  {forest_test_acc:.3f}")

importances = forest.___   # YOUR CODE — the fitted attribute holding each feature's importance
feature_names = X.columns
order = np.argsort(importances)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=importances[order], y=feature_names[order], color=PALETTE["primary"], ax=ax)
ax.set_xlabel("Importance")
ax.set_title("Random Forest Feature Importance — Heart Disease")
plt.tight_layout()
plt.show()

# --- checks ---
assert 0.0 <= forest.oob_score_ <= 1.0, \
    "oob_score_ should be a valid accuracy between 0 and 1"
assert np.isclose(importances[order][-1], importances.max()), \
    "The last (rightmost) bar in the sorted plot should correspond to the largest importance value"
print("✓ B2 complete!")

### B3 — Bagging by hand

Build a tiny random forest yourself: bootstrap-resample the training set 15 times, fit one shallow tree per resample, and combine their predictions with a majority vote. Compare the result to a single tree fit on the full training set.

In [ ]:
# B3 — Build a small bagging ensemble by hand and compare it to a single tree
rng = np.random.default_rng(42)
n_train = X_train.shape[0]
n_bootstrap_trees = 15

bootstrap_preds = []
for i in range(n_bootstrap_trees):
    sample_idx = rng.choice(___, size=n_train, replace=True)   # YOUR CODE — the number of rows in the training set (n_train)
    X_boot = X_train.iloc[sample_idx]
    y_boot = y_train.iloc[sample_idx]

    tree_i = DecisionTreeClassifier(max_depth=4, random_state=i)
    tree_i.fit(___, ___)   # YOUR CODE — this iteration's bootstrap-resampled features and target
    bootstrap_preds.append(tree_i.predict(X_test))

# Majority vote across the bootstrap trees
bootstrap_preds = np.array(bootstrap_preds)
ensemble_preds = (bootstrap_preds.mean(axis=0) >= 0.5).astype(int)
ensemble_acc = accuracy_score(___, ___)   # YOUR CODE — the true test labels and the ensemble's majority-vote predictions

single_tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)
single_tree_acc = single_tree.score(X_test, y_test)

print(f"Single tree (max_depth=4) test accuracy:          {single_tree_acc:.3f}")
print(f"Hand-built {n_bootstrap_trees}-tree bagged ensemble test accuracy: {ensemble_acc:.3f}")

# --- checks ---
assert bootstrap_preds.shape == (n_bootstrap_trees, len(y_test)), \
    "Should have one prediction row per bootstrap tree"
assert ensemble_acc >= single_tree_acc - 0.03, \
    "The bagged ensemble should be roughly as accurate as, or more accurate than, a single tree"
print("✓ B3 complete!")

---
## Section C — Applied Problem

A clinic wants to deploy a tree-based model but isn't sure how much of an unrestricted tree's accuracy is real generalization versus overfitting. Fit an unrestricted tree, a depth-tuned tree, and a random forest on the same split, then compare their train/test gaps side by side. Fill in the blanks to run the full pipeline end to end.

In [ ]:
# Section C — Comparing an unrestricted tree, a tuned tree, and a random forest

# --- Step 1: Load and split ---
X_c, y_c = load_heart_disease()
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    ___, ___, test_size=___, random_state=___, stratify=___
    # YOUR CODE — split the features and target, holding out 20% of the data
    # with a fixed seed for reproducibility, preserving the class balance
)

# --- Step 2: Fit an unrestricted-depth tree ---
unrestricted_tree = DecisionTreeClassifier(max_depth=None, random_state=42).fit(___, ___)   # YOUR CODE — the training features and training target
unrestricted_train_acc = unrestricted_tree.score(X_c_train, y_c_train)
unrestricted_test_acc = unrestricted_tree.score(___, ___)   # YOUR CODE — the test features and test target
unrestricted_gap = unrestricted_train_acc - unrestricted_test_acc

# --- Step 3: Fit a depth-tuned tree ---
tuned_tree = ___(max_depth=4, random_state=42).fit(X_c_train, y_c_train)   # YOUR CODE — the decision tree classifier for this exercise
tuned_train_acc = tuned_tree.score(X_c_train, y_c_train)
tuned_test_acc = tuned_tree.score(X_c_test, y_c_test)
tuned_gap = tuned_train_acc - tuned_test_acc

# --- Step 4: Fit a random forest ---
forest_c = ___(n_estimators=200, max_depth=4, random_state=42).fit(___, ___)   # YOUR CODE — the random forest classifier for this exercise; the training features and training target
forest_train_acc = forest_c.score(X_c_train, y_c_train)
forest_test_acc = forest_c.score(X_c_test, y_c_test)
forest_gap = forest_train_acc - forest_test_acc

# --- Step 5: Compare all three ---
results = pd.DataFrame({
    "model": ["Unrestricted Tree", "Tuned Tree (depth=4)", "Random Forest (depth=4)"],
    "train_acc": [unrestricted_train_acc, tuned_train_acc, forest_train_acc],
    "test_acc": [unrestricted_test_acc, tuned_test_acc, forest_test_acc],
    "gap": [unrestricted_gap, tuned_gap, forest_gap],
})
print(results.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=results, x="model", y="gap", color=PALETTE["secondary"], ax=ax)
ax.set_ylabel("Train − test accuracy gap")
ax.set_title("Overfitting Gap by Model")
plt.tight_layout()
plt.show()

# --- checks ---
assert unrestricted_gap == results["gap"].max(), \
    "The unrestricted-depth tree should show the largest train/test gap of the three models"
assert forest_test_acc >= tuned_test_acc - 0.05, \
    "The random forest's test accuracy should be roughly as good as, or better than, the single tuned tree's"
print("✓ Section C complete!")
print(f"  Unrestricted tree gap: {unrestricted_gap:.3f}. Tuned tree gap: {tuned_gap:.3f}. Forest gap: {forest_gap:.3f}.")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

A random forest usually beats a single shallow decision tree on accuracy, but a single shallow tree can be drawn out as an actual flowchart of yes/no questions a person can follow by hand. When would you choose the less-accurate single tree anyway? Connect your answer to `ml_concepts/13_interpretability_vs_complexity.ipynb`.

*Your response here...*

**Question D2**

In B2, your random forest's OOB score and its actual test-set accuracy were close but not identical. What should you conclude when they disagree by a noticeable amount, and would you still hold out a separate test set even though OOB scoring is "free"?

*Your response here...*

**Question D3**

Describe a dataset from your own life or work (school records, a hobby you track data for, a job you've worked) where a random forest's feature importances — telling you which input mattered most to a prediction — would actually change a real decision someone makes.

*Your response here...*